### 1997-2020 Crop Data

**Columns Description:**

1. **Crop**: The name of the crop cultivated.
2. **Crop_Year**: The year in which the crop was grown.
3. **Season**: The specific cropping season (e.g., Kharif, Rabi, Whole Year).
4. **State**: The Indian state where the crop was cultivated.
5. **Area**: The total land area (in hectares) under cultivation for the specific crop.
6. **Production**: The quantity of crop production (in metric tons).
7. **Annual_Rainfall**: The annual rainfall received in the crop-growing region (in mm).
8. **Fertilizer**: The total amount of fertilizer used for the crop (in kilograms).
9. **Pesticide**: The total amount of pesticide used for the crop (in kilograms).
10. **Yield**: The calculated crop yield (production per unit area).


In [ ]:
import pandas as pd
df = pd.read_csv('crop_yield.csv')
df.head()

,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
0,Arecanut,1997,Whole Year,Assam,73814.0,56708,2051.4,7024878.38,22882.34,0.796087
1,Arhar/Tur,1997,Kharif,Assam,6637.0,4685,2051.4,631643.29,2057.47,0.710435
2,Castor seed,1997,Kharif,Assam,796.0,22,2051.4,75755.32,246.76,0.238333
3,Coconut,1997,Whole Year,Assam,19656.0,126905000,2051.4,1870661.52,6093.36,5238.051739
4,Cotton(lint),1997,Kharif,Assam,1739.0,794,2051.4,165500.63,539.09,0.420909


In [1]:
!pip -q install ydata-profiling
from ydata_profiling import ProfileReport

profile = ProfileReport(df, title="Crop Yield Dataset — Profile Report", explorative=True)
profile.to_file("profile_report.html")
print("Saved: profile_report.html")

NameError: name 'df' is not defined

In [4]:
df.shape

(19689, 10)

In [5]:
# Remove duplicates
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]
print(f"Removed {before - after} duplicate rows")

Removed 0 duplicate rows


In [6]:
# Check info and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19689 entries, 0 to 19688
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Crop             19689 non-null  object 
 1   Crop_Year        19689 non-null  int64  
 2   Season           19689 non-null  object 
 3   State            19689 non-null  object 
 4   Area             19689 non-null  float64
 5   Production       19689 non-null  int64  
 6   Annual_Rainfall  19689 non-null  float64
 7   Fertilizer       19689 non-null  float64
 8   Pesticide        19689 non-null  float64
 9   Yield            19689 non-null  float64
dtypes: float64(5), int64(2), object(3)
memory usage: 1.5+ MB


In [7]:
df.isnull().sum()

Crop               0
Crop_Year          0
Season             0
State              0
Area               0
Production         0
Annual_Rainfall    0
Fertilizer         0
Pesticide          0
Yield              0
dtype: int64

### Missing Value Analysis
`df.isnull().sum()` shows that all columns have 0 missing values.
Therefore no imputation operation is required in this dataset.
We proceed directly to outlier handling and encoding steps.


In [8]:
print("Mean of Production:", df['Production'].mean())
print("Median of Production:", df['Production'].median())
print("Mode of Production:", df['Production'].mode()[0])

Mean of Production: 16435941.273096653
Median of Production: 13804.0
Mode of Production: 0


Mean >> Median >> Mode

The mean is much higher than the median, which usually indicates a right-skewed distribution (a few very large values are pulling the mean up).

The median (13,804) is much smaller than the mean, meaning that half the data points have production less than about 13,800.

The mode is 0, meaning that the most frequent production value is zero — so there are many entries where production was zero or no production recorded.

In [9]:
# Distribution plot of Production
import seaborn as sns
import matplotlib.pyplot as plt
sns.histplot(df['Production'], bins=30, kde=True)
plt.title("Distribution of Production")
plt.show()

C:\Users\sethu\AppData\Local\Temp\ipykernel_24596\3458854670.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
from sklearn.preprocessing import LabelEncoder

# Create a copy of the DataFrame to preserve the original
df_encoded = df.copy()

# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Apply Label Encoding
le = LabelEncoder()
for col in categorical_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col])

# Confirm encoding
print("Encoded DataFrame:")
df_encoded.head()


Encoded DataFrame:


,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
0,0,1997,4,2,73814.0,56708,2051.4,7024878.38,22882.34,0.796087
1,1,1997,1,2,6637.0,4685,2051.4,631643.29,2057.47,0.710435
2,8,1997,1,2,796.0,22,2051.4,75755.32,246.76,0.238333
3,9,1997,4,2,19656.0,126905000,2051.4,1870661.52,6093.36,5238.051739
4,11,1997,1,2,1739.0,794,2051.4,165500.63,539.09,0.420909


In [11]:
cols = df_encoded.columns
skew_table = df_encoded[cols].skew().sort_values(ascending=False)
print("Skewness of columns:\n", skew_table)

Skewness of columns:
 Pesticide          25.635746
Area               21.858218
Production         19.299193
Fertilizer         13.412599
Yield              12.785265
Annual_Rainfall     2.131785
Season              0.739242
State               0.050369
Crop_Year          -0.162656
Crop               -0.221717
dtype: float64


In [12]:
# Compute correlation matrix
corr_matrix = df_encoded.corr()
print(corr_matrix)
# Plot the heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

                     Crop  Crop_Year    Season     State      Area  \
Crop             1.000000   0.010933  0.037492  0.038587  0.071105   
Crop_Year        0.010933   1.000000 -0.053790  0.035491 -0.035686   
Season           0.037492  -0.053790  1.000000 -0.036625 -0.031369   
State            0.038587   0.035491 -0.036625  1.000000  0.026989   
Area             0.071105  -0.035686 -0.031369  0.026989  1.000000   
Production      -0.075893   0.003366  0.096856  0.003917  0.037441   
Annual_Rainfall  0.033462  -0.011187  0.099357  0.083953 -0.106054   
Fertilizer       0.074676   0.011169 -0.031800  0.026947  0.973255   
Pesticide        0.066409  -0.004657 -0.030598  0.027629  0.973479   
Yield           -0.110894   0.002539  0.141791  0.009668  0.001858   

                 Production  Annual_Rainfall  Fertilizer  Pesticide     Yield  
Crop              -0.075893         0.033462    0.074676   0.066409 -0.110894  
Crop_Year          0.003366        -0.011187    0.011169  -0.004657  

C:\Users\sethu\AppData\Local\Temp\ipykernel_24596\3657364121.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# Outlier counts per continuous column (IQR rule)
cont_cols = df_encoded.columns
outlier_summary = {}
for col in cont_cols:
    Q1, Q3 = df_encoded[col].quantile(0.25), df_encoded[col].quantile(0.75)
    IQR = Q3 - Q1
    lb, ub = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    outlier_summary[col] = int(((df_encoded[col] < lb) | (df_encoded[col] > ub)).sum())
print("Outlier counts (IQR):", outlier_summary)

Outlier counts (IQR): {'Crop': 0, 'Crop_Year': 0, 'Season': 0, 'State': 0, 'Area': 3076, 'Production': 3373, 'Annual_Rainfall': 1527, 'Fertilizer': 3093, 'Pesticide': 3036, 'Yield': 3065}


In [14]:
plt.figure(figsize=(15, 10))
for i, col in enumerate(df_encoded.columns):
    plt.subplot(2,5, i + 1)
    sns.boxplot(y=df_encoded[col])
    plt.title(f'Box plot of {col}')
plt.tight_layout()
plt.show()

C:\Users\sethu\AppData\Local\Temp\ipykernel_24596\1220555750.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# Create a copy to avoid modifying the original DataFrame
df_no_outliers = df_encoded.copy()
print(f"Shape before removing outliers: {df_no_outliers.shape}")
# Identify numerical columns again (after encoding)
#numerical_cols = df_no_outliers.select_dtypes(include=['int64', 'float64']).columns

# Remove outliers for each numerical column using IQR
for col in df_no_outliers:
    Q1 = df_no_outliers[col].quantile(0.25)
    Q3 = df_no_outliers[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_no_outliers = df_no_outliers[(df_no_outliers[col] >= lower_bound) & (df_no_outliers[col] <= upper_bound)]

# Show result
print(f"Shape after removing outliers: {df_no_outliers.shape}")

Shape before removing outliers: (19689, 10)
Shape after removing outliers: (8779, 10)


In [16]:
plt.figure(figsize=(15, 10))
for i, col in enumerate(df_no_outliers):
    plt.subplot(2,5, i + 1)
    sns.boxplot(y=df_no_outliers[col])
    plt.title(f'Box plot of {col}')
plt.tight_layout()
plt.show()

C:\Users\sethu\AppData\Local\Temp\ipykernel_24596\788129796.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# Recalculate correlation matrix without outliers
corr_matrix_no_outliers = df_no_outliers.corr()
print("Correlation Matrix (No Outliers):")
print(corr_matrix_no_outliers)
# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix_no_outliers, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap (No Outliers)")
plt.show()

Correlation Matrix (No Outliers):
                     Crop  Crop_Year    Season     State      Area  \
Crop             1.000000   0.016934  0.030719  0.059493  0.045307   
Crop_Year        0.016934   1.000000 -0.070293  0.020797 -0.150425   
Season           0.030719  -0.070293  1.000000 -0.047226  0.020682   
State            0.059493   0.020797 -0.047226  1.000000 -0.103178   
Area             0.045307  -0.150425  0.020682 -0.103178  1.000000   
Production       0.023895  -0.074658  0.103902 -0.016448  0.702809   
Annual_Rainfall  0.068287  -0.049558  0.019117  0.209069 -0.110366   
Fertilizer       0.044836  -0.020552  0.011993 -0.096084  0.973601   
Pesticide        0.055557  -0.050288  0.014696 -0.103246  0.911580   
Yield           -0.022179   0.073719  0.159740  0.071894 -0.011939   

                 Production  Annual_Rainfall  Fertilizer  Pesticide     Yield  
Crop               0.023895         0.068287    0.044836   0.055557 -0.022179  
Crop_Year         -0.074658        

C:\Users\sethu\AppData\Local\Temp\ipykernel_24596\2660409406.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df_no_outliers.drop('Production', axis=1)
y = df_no_outliers['Production']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

# Separate features (X) and target (y)
X = df_no_outliers.drop('Production', axis=1)
y = df_no_outliers['Production']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the models to be evaluated
models = {
    "Random Forest": RandomForestRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "K-Nearest Neighbors": KNeighborsRegressor()
}

# Dictionary to store results
results = {}

# Loop through the models, train, and evaluate
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    results[name] = [r2, rmse, mae]
    print(f"--- {name} ---")
    print(f"R² Score: {r2:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}\n")

--- Random Forest ---
R² Score: 0.96
RMSE: 1437.29
MAE: 434.24

--- XGBoost ---
R² Score: 0.96
RMSE: 1405.23
MAE: 479.03

--- AdaBoost ---
R² Score: 0.72
RMSE: 3658.51
MAE: 3076.57

--- K-Nearest Neighbors ---
R² Score: 0.34
RMSE: 5641.49
MAE: 2631.84



### 📈 Model Performance Summary

| Model                 | R² Score | RMSE   | MAE    |
|-----------------------|----------|--------|--------|
| Random Forest         | 0.96     | 1437   | 434    |
| **XGBoost (Best)**    | **0.96**| **1405**| **479**|
| AdaBoost              | 0.72    | 3658   | 3076   |
| KNN                   | 0.34    | 5641   | 2632   |


In [20]:
from sklearn.preprocessing import (
    MinMaxScaler, StandardScaler, RobustScaler, Normalizer,
    QuantileTransformer, PowerTransformer
)
from sklearn.pipeline import Pipeline

# Define the scalers
scalers = {
    "Min-Max": MinMaxScaler(),
    "Z-Score (Standard)": StandardScaler(),
    "Robust Scaling": RobustScaler(),
    "L2 Normalization": Normalizer(norm='l2'),
    "Quantile Uniform": QuantileTransformer(output_distribution='uniform'),
    "Quantile Normal": QuantileTransformer(output_distribution='normal'),
    "Power Transformer": PowerTransformer(method='yeo-johnson'),
    "Log Transform": 'log'
}

# Store results for scaling comparison
scaling_results = {}

# Choose the best model
best_model = XGBRegressor(random_state=42)

# Loop through each scaler
for name, scaler in scalers.items():
    if name == "Log Transform":
        # Apply log transform directly
        X_train_log = np.log1p(X_train)
        X_test_log = np.log1p(X_test)
        best_model.fit(X_train_log, y_train)
        y_pred = best_model.predict(X_test_log)
    else:
        # Create a pipeline with the scaler and the model
        pipeline = Pipeline([
            ('scaler', scaler),
            ('model', best_model)
        ])
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    scaling_results[name] = [r2, rmse]

# Add the baseline (no scaling) result for comparison
scaling_results["No Scaling (Baseline)"] = results["XGBoost"][:2]

# Display results
scaling_results_df = pd.DataFrame(scaling_results, index=['R² Score', 'RMSE']).T.sort_values(by='R² Score', ascending=False)
scaling_results_df

c:\Users\sethu\anaconda3\Lib\site-packages\numpy\core\_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)


,R² Score,RMSE
Min-Max,0.959012,1405.232155
Z-Score (Standard),0.959012,1405.232155
Robust Scaling,0.959012,1405.232155
Quantile Uniform,0.959012,1405.232155
Quantile Normal,0.959012,1405.232155
Power Transformer,0.959012,1405.232155
Log Transform,0.959012,1405.232155
No Scaling (Baseline),0.959012,1405.232155
L2 Normalization,0.952498,1512.792729


### Skewness Interpretation
Skewness values show extremely high right skew for `Pesticide (25.63)`, `Area (21.85)`, `Production (19.29)`, `Fertilizer (13.41)` and `Yield (12.78)`.  
This means most observations have very small values while few very large values stretch the tail.

Such heavy right skew often occurs in agricultural datasets because only few states/crops have massive scale production while majority are small farmers.

Since tree based models are robust to skewness, transformation is optional for them.
But PCA + linear models + KNN benefit from normalization + scaling before modeling.

In [21]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_full = df_encoded.drop(columns=['Production'])  # target remains 'Production'
y_full = df_encoded['Production']

X_scaled = StandardScaler().fit_transform(X_full)  # PCA prefers standardized data
pca = PCA(n_components=0.95, svd_solver='full')   # keep 95% variance
X_pca = pca.fit_transform(X_scaled)
print("PCA components:", pca.n_components_, "Explained variance:", pca.explained_variance_ratio_.sum())

PCA components: 7 Explained variance: 0.9928657683623361


In [22]:
from sklearn.model_selection import cross_val_score, KFold

# Cross-validation on best model
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rmse = (-cross_val_score(models["XGBoost"], X, y, scoring='neg_root_mean_squared_error', cv=cv)).mean()
cv_r2 = (cross_val_score(models["XGBoost"], X, y, scoring='r2', cv=cv)).mean()
print(f"XGB 5-fold CV — RMSE: {cv_rmse:.2f}, R²: {cv_r2:.3f}")


XGB 5-fold CV — RMSE: 1436.43, R²: 0.964


In [23]:
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor())
])

knn_pipeline.fit(X_train, y_train)
knn_pred = knn_pipeline.predict(X_test)

knn_r2 = r2_score(y_test, knn_pred)
knn_rmse = np.sqrt(mean_squared_error(y_test, knn_pred))
knn_mae = mean_absolute_error(y_test, knn_pred)

print("KNN with StandardScaler (Normalized)")
print("R2:", knn_r2)
print("RMSE:", knn_rmse)
print("MAE:", knn_mae)


KNN with StandardScaler (Normalized)
R2: 0.9124151733641642
RMSE: 2054.1681185010293
MAE: 828.6973804100228


In [24]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
import numpy as np

models_compare = {}

# Separate features and target for the data without outliers
X_no_outliers = df_no_outliers.drop('Production', axis=1)
y_no_outliers = df_no_outliers['Production']

# XGB Original (using data without outliers)
X_train_orig, X_test_orig, y_train_orig, y_test_orig = train_test_split(X_no_outliers, y_no_outliers, test_size=0.2, random_state=42)
xgb_orig = XGBRegressor(random_state=42)
xgb_orig.fit(X_train_orig, y_train_orig)
pred_orig = xgb_orig.predict(X_test_orig)
models_compare["XGB Original"] = [
    r2_score(y_test_orig, pred_orig),
    np.sqrt(mean_squared_error(y_test_orig, pred_orig)),
    mean_absolute_error(y_test_orig, pred_orig)
]

# PCA on the data without outliers
X_scaled_no_outliers = StandardScaler().fit_transform(X_no_outliers)
pca = PCA(n_components=0.95, svd_solver='full')
X_pca_no_outliers = pca.fit_transform(X_scaled_no_outliers)
print("PCA components:", pca.n_components_, "Explained variance:", pca.explained_variance_ratio_.sum())

# Train-test split for PCA data (from data without outliers)
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(X_pca_no_outliers, y_no_outliers, test_size=0.2, random_state=42)

# XGB PCA
xgb_pca = XGBRegressor(random_state=42)
xgb_pca.fit(X_train_pca, y_train_pca)
pred_pca = xgb_pca.predict(X_test_pca)
models_compare["XGB PCA"] = [
    r2_score(y_test_pca, pred_pca),
    np.sqrt(mean_squared_error(y_test_pca, pred_pca)),
    mean_absolute_error(y_test_pca, pred_pca)
]

compare_df = pd.DataFrame(models_compare, index=["R2","RMSE","MAE"]).T
compare_df

PCA components: 7 Explained variance: 0.984760532788746


,R2,RMSE,MAE
XGB Original,0.959012,1405.232155,479.029267
XGB PCA,0.934573,1775.421523,733.973570


# Model Performance Interpretation

---

### Random Forest
- **R² Score:** 0.96  
- **RMSE:** 1437.29  
The Random Forest model performs very well, explaining 96% of the variance in the target variable. The RMSE value indicates a relatively low average prediction error, showing that this model provides accurate predictions.

---

### XGBoost
- **R² Score:** 0.96  
- **RMSE:** 1405.23  
XGBoost also achieves an excellent performance, matching the Random Forest in R² and slightly outperforming it in RMSE. This suggests that XGBoost has a slightly better fit and lower prediction error than Random Forest.

---

### AdaBoost
- **R² Score:** 0.72  
- **RMSE:** 3658.51  
AdaBoost’s performance is noticeably lower than Random Forest and XGBoost. An R² of 0.72 means it explains 72% of the variance, which is decent but less reliable. The higher RMSE indicates larger average prediction errors.

---

### K-Nearest Neighbors (KNN)
- **R² Score:** 0.34  
- **RMSE:** 5641.49  
KNN shows the weakest performance with only 34% of the variance explained. The high RMSE reflects significant errors in predictions, suggesting KNN is not well-suited for this problem.

---

## Summary:
- **Random Forest** and **XGBoost** are the best-performing models, with very high R² scores and low RMSE values.  
- **AdaBoost** is moderately effective but less accurate.  
- **K-Nearest Neighbors** performs poorly for this task and likely should not be used in production.

Based on these metrics, **XGBoost** might be preferred for slightly better accuracy.

In [25]:
import joblib

joblib.dump(xgb_orig, 'crop_yield.pkl')

print("Model saved as crop_yield.pkl")

Model saved as crop_yield.pkl


In [1]:
import xgboost as xgb
print(xgb.__version__)

3.0.1


In [3]:
import sys
print(f"Python: {sys.version}")
print("\n" + "="*50)
print("Key Libraries:")
print("="*50)

import pandas as pd
import numpy as np
import xgboost as xgb
import sklearn
import matplotlib
import seaborn

libs = {
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "xgboost": xgb.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": seaborn.__version__,
}

for lib, version in libs.items():
    print(f"{lib}: {version}")

Python: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]

Key Libraries:
pandas: 2.2.2
numpy: 1.26.4
xgboost: 3.0.1
scikit-learn: 1.5.1
matplotlib: 3.9.2
seaborn: 0.13.2
